# AI-асистент продуктового аналітика: демо на реальних даних

Лабораторна робота №4 з МІПА, Бельдяга Назар, ІП-31.

Ідея: **pandas рахує, LLM пояснює.** Усі числа (профіль даних, сегменти, внески, p-value) рахує код,
а Gemini отримує їх у вигляді фактів і пише висновки. Після кожної відповіді код перевіряє, чи немає
в тексті моделі чисел, яких не було у фактах.

Датасет: мобільний застосунок знайомств, 5.04-11.05.2024 (реєстрації, 1,5 млн сесій, платежі).

## 0. Підготовка
Клоную репозиторій і ставлю залежності. Ключ Gemini береться з Colab Secrets (`GEMINI_API_KEY`).

In [ ]:
%cd /content
!rm -rf mipa-lab4-product-ai && git clone -q https://github.com/nazarbeldyaga/mipa-lab4-product-ai.git
%cd mipa-lab4-product-ai
!pip install -q -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass  # локально ключ задається змінною середовища

from product_ai import Analyst, GeminiLLM, load_data, show
import pandas as pd

data = load_data()
for name, df in data.items():
    print(f"{name:10s} {len(df):>9,} рядків, {df.shape[1]} колонок")

In [ ]:
llm = GeminiLLM()
print("Модель:", llm.model)
ai = Analyst(data, llm)

## 1. Автоматичний EDA
Профіль рахує pandas (`profile_table`), модель оцінює якість і пояснює розподіли.

In [ ]:
res = ai.eda("sessions")
show(res)

In [ ]:
res = ai.eda("payments")
show(res)

## 2. Генерація гіпотез
pandas рахує 11 «сигналів» з id (F01...), модель вибирає найцікавіші і формулює гіпотези з посиланням на факти.

In [ ]:
res_h = ai.hypotheses(n=7)
show(res_h)

In [ ]:
# факти, на які посилаються гіпотези (щоб можна було перевірити руками)
pd.DataFrame([{"id": f["id"], "topic": f["topic"]} for f in res_h["facts"]])

## 3. Питання природною мовою → pandas
Код показується разом з результатом. Перед запуском його перевіряє `check_code` (без імпортів, файлів, eval).

In [ ]:
show(ai.ask("Покажи топ-10 країн за виручкою у квітні 2024 і кількість платників у кожній"))

In [ ]:
show(ai.ask("Яка середня тривалість сесії в хвилинах по вікових групах для iOS і Mobile web?"))

In [ ]:
# нелогічне питання: у даних немає такої інформації
show(ai.ask("Яка була погода в Києві в день з найбільшою виручкою?"))

## 4. Пояснення зміни метрики
Активних користувачів за тиждень 4-10 травня стало помітно менше, ніж тижнем раніше. Розкладаю зміну за платформою, країною, статтю і віком.

In [ ]:
res_c = ai.explain("active_users", base=("2024-04-27", "2024-05-03"), target=("2024-05-04", "2024-05-10"))
show(res_c)

In [ ]:
# той самий розклад, але для метрики-відношення: частка платників серед активних (mix + rate)
show(ai.explain("payer_conversion", base=("2024-04-27", "2024-05-03"), target=("2024-05-04", "2024-05-10")))

## 5. Executive summary

In [ ]:
show(ai.summary("2024-05-04", "2024-05-10"))